# Ranking Engine (work/03_ranking_engine)

This notebook builds a simple rule-based ranking score and evaluates precision@K using the labelled aggregate. If you later build a model, put training code here and compare to the rule baseline.


In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
Path('work/outputs').mkdir(parents=True, exist_ok=True)
if os.path.exists('work/outputs/labelled_agg.parquet'):
    df = pd.read_parquet('work/outputs/labelled_agg.parquet')
elif os.path.exists('work/outputs/eda_starter.parquet'):
    df = pd.read_parquet('work/outputs/eda_starter.parquet')
else:
    raise FileNotFoundError('No cached input found. Run the starter EDA or the warehouse aggregation step.')

# Simple score example (weights may be tuned)
df['is_low_position'] = (df['avg_position'] > 10).fillna(0).astype(int) if 'avg_position' in df.columns else 0
df['has_keywords'] = df.get('has_keywords', 0)
df['is_low_wordcount'] = (df.get('word_count', 0) < 300).astype(int) if 'word_count' in df.columns else 0
df['low_ctr_label'] = df.get('low_ctr_label', 0)
df['score'] = 3*df['is_low_position'] + 2*df['low_ctr_label'] + 1*(1 - df['has_keywords']) + 0.5*df['is_low_wordcount']
df['score_norm'] = (df['score'] - df['score'].min()) / (df['score'].max() - df['score'].min()) if df['score'].max() != df['score'].min() else 0

# Precision@K function
def precision_at_k(df_sorted, label_col='decline_label', k=10):
    topk = df_sorted.head(k)
    if label_col not in topk.columns:
        return None
    return topk[label_col].sum() / k

for k in [5,10,20]:
    print('Precision@',k, precision_at_k(df.sort_values('score_norm', ascending=False), 'low_ctr_label', k))

# Save top recommendations
top = df.sort_values('score_norm', ascending=False).head(500)
top.to_csv('work/outputs/top_500_recommendations.csv', index=False)
print('Saved top_500_recommendations.csv')
